In [1]:
# 테스트용 토픽 생성
from confluent_kafka.admin import AdminClient, NewTopic

admin = AdminClient({"bootstrap.servers": "kafka:9092"})

# 토픽 설정: 4개 파티션, 복제 1
topic = NewTopic(topic="api-events", num_partitions=4, replication_factor=1)

# 토픽 생성
fs = admin.create_topics([topic])

# 결과 확인
for topic_name, f in fs.items():
    try:
        f.result()  # 완료 대기
        print(f"토픽 '{topic_name}' 생성 완료!")
    except Exception as e:
        print(f"토픽 '{topic_name}' 생성 실패: {e}")

# 토픽 목록 확인
metadata = admin.list_topics(timeout=10)
print(
    f"\n현재 토픽 목록: {[t for t in metadata.topics.keys() if not t.startswith('_')]}"
)

토픽 'api-events' 생성 완료!

현재 토픽 목록: ['api-events']


In [2]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("ConnectionTest")\
.master("spark://spark-master:7077")\
.config("spark.executor.memory","1g")\
.config("spark.executor.cores","1")\
.getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 02:10:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/01/19 02:10:31 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
df = spark.range(1000)
df

DataFrame[id: bigint]

In [6]:
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
| 10|
| 11|
| 12|
| 13|
| 14|
| 15|
| 16|
| 17|
| 18|
| 19|
+---+
only showing top 20 rows



In [7]:
spark.stop()

In [ ]:
## <코드정리>
## API 서버에서 발생한 요청 로그처럼 생긴 이벤트를 랜덤으로 생성해서 
## Kafka 토픽(api-events)으로 10건 전송하는 Producer 코드

# Step 1: Producer 빈칸 채우기
from confluent_kafka import Producer
import json
import random
from datetime import datetime

# -----------------------------------------------------------------------------
# Producer 설정
# -----------------------------------------------------------------------------
config = {
    # TODO 1: Kafka 브로커 주소를 설정하세요
    # 힌트: Docker 서비스 이름과 포트
    "bootstrap.servers": "kafka:9092",
    "client.id": "api-event-producer",
}

# TODO 2: Producer 객체 생성 
## 아직 메시지 안보냄(내부적으로 > 네트워크 연결 준비/전송 버퍼 생성/Kafka 프로토콜 초기화)
producer = Producer(config)

# -----------------------------------------------------------------------------
# API 이벤트 생성 함수
# -----------------------------------------------------------------------------
ENDPOINTS = [
    "/api/products",
    "/api/users",
    "/api/orders",
    "/api/payments",
    "/api/search",
]
METHODS = ["GET", "POST", "PUT", "DELETE"]
STATUS_CODES = [200, 200, 200, 200, 201, 400, 404, 500]  # 200이 더 자주 발생


## 실제 API 로그와 비슷한 JSON 데이터를 랜덤으로 만듦
def generate_api_event():
    """API 이벤트 데이터 생성"""
    return {
        "request_id": f"REQ_{random.randint(1, 999999):06d}",
        "user_id": f"U{random.randint(1, 1000):04d}",
        "endpoint": random.choice(ENDPOINTS),
        "method": random.choice(METHODS),
        "status_code": random.choice(STATUS_CODES),
        "response_time_ms": random.randint(10, 500),
        "timestamp": datetime.now().isoformat(),
    }


# -----------------------------------------------------------------------------
# 메시지 전송
# -----------------------------------------------------------------------------
# TODO 3: 토픽 이름 설정
TOPIC = "api-events"

## 10개의 이벤트 생성해서 Kafka로 전송
for i in range(10):
    event = generate_api_event()

    # TODO 4: 메시지 전송 (topic, value 파라미터 사용)
    ## 실제 Kafka 전송 코드(핵심)
    producer.produce(
        topic=TOPIC,
        value=json.dumps(event).encode("utf-8"),
    )
    print(f"전송: {event['request_id']} - {event['endpoint']}")

# TODO 5: 버퍼의 모든 메시지를 전송하고 대기
producer.flush()  ## 이 줄이 없으면 메시지 사라질 수 있음
# flush(): 내부 버퍼에 남아있는 메시지를 모두 Kafka로 전송할 때까지 대기
print("전송 완료!")

전송: REQ_467353 - /api/search
전송: REQ_609684 - /api/users
전송: REQ_331629 - /api/users
전송: REQ_170630 - /api/products
전송: REQ_726128 - /api/orders
전송: REQ_456845 - /api/users
전송: REQ_080206 - /api/payments
전송: REQ_904760 - /api/orders
전송: REQ_267014 - /api/orders
전송: REQ_339745 - /api/products
전송 완료!


### 코드 실행 순서도
[ 프로그램 시작 ]   
        ↓   
[ Producer 설정 ]   
        ↓     
[ Producer 객체 생성 ]   
        ↓  
[ for 루프 시작 ]   
        ↓   
[ 이벤트 생성 ]   
        ↓   
[ JSON 직렬화 + bytes 변환 ]    
        ↓    
[ producer.produce() → 버퍼 적재 ]   
        ↓   
[ print 로그 출력 ]   
        ↓  
(10번 반복)   
        ↓   
[ producer.flush() ]   
        ↓   
[ 모든 메시지 Kafka 전송 완료 ]   
        ↓   
[ 프로그램 종료 ]   


In [10]:
# Step 2: 완성된 Producer
from confluent_kafka import Producer
import json
import random
from datetime import datetime
import time

# -----------------------------------------------------------------------------
# Producer 설정
# -----------------------------------------------------------------------------
config = {
    "bootstrap.servers": "kafka:9092",
    "client.id": "api-event-producer",
}

producer = Producer(config)

# -----------------------------------------------------------------------------
# API 이벤트 생성 함수
# -----------------------------------------------------------------------------
ENDPOINTS = [
    "/api/products",
    "/api/users",
    "/api/orders",
    "/api/payments",
    "/api/search",
]
METHODS = ["GET", "POST", "PUT", "DELETE"]
STATUS_CODES = [200, 200, 200, 200, 201, 400, 404, 500]


def generate_api_event():
    """API 이벤트 데이터 생성"""
    return {
        "request_id": f"REQ_{random.randint(1, 999999):06d}",
        "user_id": f"U{random.randint(1, 1000):04d}",
        "endpoint": random.choice(ENDPOINTS),
        "method": random.choice(METHODS),
        "status_code": random.choice(STATUS_CODES),
        "response_time_ms": random.randint(10, 500),
        "timestamp": datetime.now().isoformat(),
    }


# -----------------------------------------------------------------------------
# Delivery Callback (전송 결과 확인)
# -----------------------------------------------------------------------------
sent_count = 0


def delivery_callback(err, msg):
    global sent_count
    if err:
        print(f"전송 실패: {err}")
    else:
        sent_count += 1


# -----------------------------------------------------------------------------
# 메시지 대량 전송
# -----------------------------------------------------------------------------
TOPIC = "api-events"
NUM_MESSAGES = 1000

print(f"API 이벤트 {NUM_MESSAGES}건 전송 시작...")
start_time = time.time()

for i in range(NUM_MESSAGES):
    event = generate_api_event()

    producer.produce(
        topic=TOPIC,
        value=json.dumps(event).encode("utf-8"),
        callback=delivery_callback,
    )

    # 1000건마다 진행 상황 출력
    if (i + 1) % 1000 == 0:
        producer.flush()
        print(f"  {i + 1}건 전송 완료")

producer.flush()
elapsed = time.time() - start_time

print("\n전송 완료!")
print(f"  총 전송: {sent_count}건")
print(f"  소요 시간: {elapsed:.2f}초")
print(f"  처리량: {sent_count / elapsed:.0f} records/sec")

API 이벤트 1000건 전송 시작...
  1000건 전송 완료

전송 완료!
  총 전송: 1000건
  소요 시간: 0.12초
  처리량: 8665 records/sec


In [11]:
# Step 1: Consumer 빈칸 채우기
from confluent_kafka import Consumer
import json

# -----------------------------------------------------------------------------
# Consumer 설정
# -----------------------------------------------------------------------------
config = {
    # TODO 1: Kafka 브로커 주소
    "bootstrap.servers": "kafka:9092",
    # TODO 2: Consumer Group ID (같은 그룹의 Consumer는 파티션을 나눠 처리)
    "group.id": "api-event-consumer-group",
    # TODO 3: 오프셋 설정 (earliest: 처음부터, latest: 최신부터)
    "auto.offset.reset": "earliest",
    "enable.auto.commit": True,
}

# TODO 4: Consumer 객체 생성
consumer = Consumer(config)

# TODO 5: 토픽 구독
consumer.subscribe(["api-events"])

# -----------------------------------------------------------------------------
# 메시지 수신
# -----------------------------------------------------------------------------
print("메시지 수신 대기 중... (Ctrl+C로 종료)")

try:
    count = 0
    while count < 10:  # 10개만 수신하고 종료
        # TODO 6: 메시지 가져오기 (timeout=1.0)
        msg = consumer.poll(timeout=1.0)

        if msg is None:
            continue
        if msg.error():
            print(f"에러: {msg.error()}")
            continue

        # 메시지 처리
        event = json.loads(msg.value().decode("utf-8"))
        print(
            f"수신: {event['request_id']} - {event['endpoint']} ({event['status_code']})"
        )
        count += 1

finally:
    consumer.close()

메시지 수신 대기 중... (Ctrl+C로 종료)
수신: REQ_491439 - /api/orders (200)
수신: REQ_467353 - /api/search (200)
수신: REQ_609684 - /api/users (200)
수신: REQ_873196 - /api/orders (200)
수신: REQ_418529 - /api/orders (404)
수신: REQ_513632 - /api/users (200)
수신: REQ_241197 - /api/users (400)
수신: REQ_078585 - /api/products (200)
수신: REQ_096505 - /api/payments (200)
수신: REQ_379532 - /api/payments (200)


In [12]:
# Step 2: 완성된 Consumer
from confluent_kafka import Consumer
import json
import time

# -----------------------------------------------------------------------------
# Consumer 설정
# -----------------------------------------------------------------------------
config = {
    "bootstrap.servers": "kafka:9092",
    "group.id": "api-event-consumer-group",
    "auto.offset.reset": "earliest",  # 처음부터 읽기
    "enable.auto.commit": True,
}

consumer = Consumer(config)
consumer.subscribe(["api-events"])

# -----------------------------------------------------------------------------
# 메시지 수신 및 통계
# -----------------------------------------------------------------------------
print("메시지 수신 시작...\n")

stats = {"total": 0, "by_status": {}, "by_endpoint": {}}
start_time = time.time()
max_messages = 100  # 100개 메시지 수신 후 통계 출력

try:
    while stats["total"] < max_messages:
        msg = consumer.poll(timeout=1.0)

        if msg is None:
            continue
        if msg.error():
            print(f"에러: {msg.error()}")
            continue

        # 메시지 파싱
        event = json.loads(msg.value().decode("utf-8"))

        # 통계 업데이트
        stats["total"] += 1

        status = event["status_code"]
        stats["by_status"][status] = stats["by_status"].get(status, 0) + 1

        endpoint = event["endpoint"]
        stats["by_endpoint"][endpoint] = stats["by_endpoint"].get(endpoint, 0) + 1

        # 10개마다 진행 상황
        if stats["total"] % 100 == 0:
            print(f"  수신: {stats['total']}건")

finally:
    consumer.close()

# -----------------------------------------------------------------------------
# 통계 출력
# -----------------------------------------------------------------------------
elapsed = time.time() - start_time

print(f"\n{'=' * 50}")
print(f"총 수신: {stats['total']}건 ({elapsed:.2f}초)")
print(f"처리량: {stats['total'] / elapsed:.0f} records/sec")

print(f"\n상태 코드별 분포:")
for status, count in sorted(stats["by_status"].items()):
    pct = count / stats["total"] * 100
    print(f"  {status}: {count}건 ({pct:.1f}%)")

print(f"\n엔드포인트별 분포:")
for endpoint, count in sorted(stats["by_endpoint"].items(), key=lambda x: -x[1]):
    pct = count / stats["total"] * 100
    print(f"  {endpoint}: {count}건 ({pct:.1f}%)")

메시지 수신 시작...

  수신: 100건

총 수신: 100건 (1.79초)
처리량: 56 records/sec

상태 코드별 분포:
  200: 47건 (47.0%)
  201: 14건 (14.0%)
  400: 10건 (10.0%)
  404: 16건 (16.0%)
  500: 13건 (13.0%)

엔드포인트별 분포:
  /api/orders: 31건 (31.0%)
  /api/users: 19건 (19.0%)
  /api/search: 19건 (19.0%)
  /api/products: 16건 (16.0%)
  /api/payments: 15건 (15.0%)


In [16]:
# 처리량 측정 실험
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import Producer
import json
import time
import random
from datetime import datetime

admin = AdminClient({"bootstrap.servers": "kafka:9092"})


# -----------------------------------------------------------------------------
# 토픽 생성 함수
# -----------------------------------------------------------------------------
def create_topic(name, num_partitions):
    """토픽 생성 (이미 존재하면 삭제 후 생성)"""
    # 기존 토픽 삭제 시도
    try:
        admin.delete_topics([name]).get(name).result()
        time.sleep(2)  # 삭제 완료 대기
    except:
        pass

    # 새 토픽 생성
    topic = NewTopic(topic=name, num_partitions=num_partitions, replication_factor=1)
    fs = admin.create_topics([topic])
    fs[name].result()
    time.sleep(1)
    print(f"토픽 '{name}' 생성 (파티션: {num_partitions})")


# -----------------------------------------------------------------------------
# 처리량 측정 함수
# -----------------------------------------------------------------------------
def measure_throughput(topic_name, num_messages):
    """메시지 전송 처리량 측정"""
    producer = Producer({"bootstrap.servers": "kafka:9092"})

    sent_count = 0

    def callback(err, msg):
        nonlocal sent_count
        if not err:
            sent_count += 1

    start_time = time.time()

    for i in range(num_messages):
        event = {
            "request_id": f"REQ_{i:06d}",
            "user_id": f"U{random.randint(1, 1000):04d}",
            "endpoint": f"/api/test",
            "timestamp": datetime.now().isoformat(),
        }

        producer.produce(
            topic=topic_name,
            value=json.dumps(event).encode("utf-8"),
            callback=callback,
        )

        # 주기적으로 flush
        if (i + 1) % 10000 == 0:
            producer.flush()

    producer.flush()
    elapsed = time.time() - start_time

    return sent_count, elapsed


# -----------------------------------------------------------------------------
# 실험 실행
# -----------------------------------------------------------------------------
NUM_MESSAGES = 50000

print("=" * 60)
print("파티션 수에 따른 처리량 비교")
print("=" * 60)

# 실험 1: 파티션 1개
create_topic("throughput-test-1p", 1)
count1, time1 = measure_throughput("throughput-test-1p", NUM_MESSAGES)
throughput1 = count1 / time1
print(f"  파티션 1개: {throughput1:,.0f} records/sec ({time1:.2f}초)")

# 실험 2: 파티션 4개
create_topic("throughput-test-4p", 4)
count4, time4 = measure_throughput("throughput-test-4p", NUM_MESSAGES)
throughput4 = count4 / time4
print(f"  파티션 4개: {throughput4:,.0f} records/sec ({time4:.2f}초)")

# 비교
print(f"\n결과:")
print(f"  처리량 향상: {(throughput4 / throughput1 - 1) * 100:.1f}%")

파티션 수에 따른 처리량 비교
토픽 'throughput-test-1p' 생성 (파티션: 1)
  파티션 1개: 214,066 records/sec (0.23초)
토픽 'throughput-test-4p' 생성 (파티션: 4)
  파티션 4개: 211,213 records/sec (0.24초)

결과:
  처리량 향상: -1.3%


In [3]:
# Producer: 파티션 4개 토픽에 100만 건 전송
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic
import json
import random
from datetime import datetime
import time

# -----------------------------------------------------------------------------
# 토픽 생성 (파티션 4개)
# -----------------------------------------------------------------------------
TOPIC = "api-events"

admin = AdminClient({"bootstrap.servers": "kafka:9092"})

# 기존 토픽 삭제 후 재생성
try:
    admin.delete_topics([TOPIC])[TOPIC].result()
    time.sleep(2)
    print(f"기존 토픽 '{TOPIC}' 삭제")
except:
    pass

topic = NewTopic(topic=TOPIC, num_partitions=4, replication_factor=1)
admin.create_topics([topic])[TOPIC].result()
print(f"토픽 '{TOPIC}' 생성 (파티션: 4개)\n")

# -----------------------------------------------------------------------------
# Producer 설정
# -----------------------------------------------------------------------------
config = {
    "bootstrap.servers": "kafka:9092",
    "client.id": "api-event-producer",
}

producer = Producer(config)

# -----------------------------------------------------------------------------
# API 이벤트 생성 함수
# -----------------------------------------------------------------------------
ENDPOINTS = ["/api/products", "/api/users", "/api/orders", "/api/payments", "/api/search"]
METHODS = ["GET", "POST", "PUT", "DELETE"]
STATUS_CODES = [200, 200, 200, 200, 201, 400, 404, 500]


def generate_api_event():
    return {
        "request_id": f"REQ_{random.randint(1, 999999):06d}",
        "user_id": f"U{random.randint(1, 1000):04d}",
        "endpoint": random.choice(ENDPOINTS),
        "method": random.choice(METHODS),
        "status_code": random.choice(STATUS_CODES),
        "response_time_ms": random.randint(10, 500),
        "timestamp": datetime.now().isoformat(),
    }


# -----------------------------------------------------------------------------
# Delivery Callback
# -----------------------------------------------------------------------------
sent_count = 0


def delivery_callback(err, msg):
    global sent_count
    if err:
        print(f"전송 실패: {err}")
    else:
        sent_count += 1


# -----------------------------------------------------------------------------
# 메시지 100만 건 전송
# -----------------------------------------------------------------------------
NUM_MESSAGES = 1000000

print(f"API 이벤트 {NUM_MESSAGES:,}건 전송 시작...")
start_time = time.time()

for i in range(NUM_MESSAGES):
    event = generate_api_event()

    producer.produce(
        topic=TOPIC,
        value=json.dumps(event).encode("utf-8"),
        callback=delivery_callback,
    )

    if (i + 1) % 100000 == 0:
        producer.flush()
        print(f"  {i + 1:,}건 전송 완료")

producer.flush()
elapsed = time.time() - start_time

print("\n전송 완료!")
print(f"  총 전송: {sent_count:,}건")
print(f"  소요 시간: {elapsed:.2f}초")
print(f"  처리량: {sent_count / elapsed:,.0f} records/sec")

기존 토픽 'api-events' 삭제
토픽 'api-events' 생성 (파티션: 4개)

API 이벤트 1,000,000건 전송 시작...
  100,000건 전송 완료
  200,000건 전송 완료
  300,000건 전송 완료
  400,000건 전송 완료
  500,000건 전송 완료
  600,000건 전송 완료
  700,000건 전송 완료
  800,000건 전송 완료
  900,000건 전송 완료
  1,000,000건 전송 완료

전송 완료!
  총 전송: 1,000,000건
  소요 시간: 6.36초
  처리량: 157,137 records/sec


In [4]:
# Consumer: 병렬 처리 (노트북 2개에서 동시 실행)
from confluent_kafka import Consumer
import json
import time

config = {
    "bootstrap.servers": "kafka:9092",
    "group.id": "parallel-consumer-group",  # 같은 그룹 ID!
    "auto.offset.reset": "earliest",
}

consumer = Consumer(config)
consumer.subscribe(["api-events"])

print("Consumer 시작...")
count = 0
start = None
last_msg_time = None

empty_count = 0
max_empty = 3  # 1초 x 3번 = 3초 대기 후 종료
first_message = True

try:
    while True:
        # 첫 메시지는 10초 대기, 이후 1초 대기
        timeout = 10.0 if first_message else 1.0
        msg = consumer.poll(timeout)

        if msg is None:
            empty_count += 1
            if first_message:
                print("10초 동안 메시지 없음. 종료합니다.")
                break
            if empty_count >= max_empty:
                print("더 이상 메시지 없음. 종료합니다.")
                break
            continue

        if msg.error():
            continue

        empty_count = 0

        # 첫 메시지 수신 시 시간 측정 시작
        if first_message:
            start = time.time()
            first_message = False

        last_msg_time = time.time()
        count += 1

        if count % 100000 == 0:
            print(f"  {count:,}건 처리중 (파티션 {msg.partition()})")

except KeyboardInterrupt:
    pass
finally:
    consumer.close()

# 결과 출력 (대기 시간 제외, 실제 처리 시간만 측정)
print(f"\n{'=' * 50}")
if count > 0 and start and last_msg_time:
    elapsed = last_msg_time - start
    print(f"총 수신: {count:,}건")
    print(f"소요 시간: {elapsed:.2f}초 (대기 시간 제외)")
    print(f"처리량: {count / elapsed:,.0f} records/sec")
else:
    print("수신된 메시지 없음")

Consumer 시작...
  100,000건 처리중 (파티션 1)
  200,000건 처리중 (파티션 1)
  300,000건 처리중 (파티션 1)
  400,000건 처리중 (파티션 0)
더 이상 메시지 없음. 종료합니다.

총 수신: 483,755건
소요 시간: 3.24초 (대기 시간 제외)
처리량: 149,263 records/sec
